# Vaani Track 1 - one-off data preparation

Run this **once**. It downloads the corpus, labels speech with Silero VAD,
builds the synthetic bronze clips, and packs all of it into a handful of large
files under `/kaggle/working/vaani`. Then *Save Version* - the output is what
the training notebook attaches as an input.

Why a separate notebook: a Kaggle notebook's saved output is capped at **500
files**, and the prepared corpus is ~110k of them. Without packing, nothing
survives the session, and every 12 h training session spent its first hour
re-downloading 16.5 GB and re-running VAD and synthesis.

Settings: *Accelerator -> GPU T4 x2* (Silero VAD runs on the GPU), *Internet ->
On*. The dataset is gated: paste an `HF_TOKEN` below or attach the Kaggle secret.

Expected: ~12-13 GB of output in about 10 files. The unpacked working copy lives
in `/tmp`, which is not saved and not part of the 20 GB output quota.


In [ ]:
HF_TOKEN = ""   # paste a Hugging Face read token, or leave "" to use the HF_TOKEN secret


In [ ]:
# ============================== CONFIG ==============================
REPO        = "raut7218/vaani-sed-v2"
BRANCH      = "speed"
N_SYNTHETIC = 20000
MAX_SHARDS  = 0                  # 0 = all 182 shards
OUT         = "/kaggle/working/vaani"
SCRATCH     = "/tmp/vaani"       # unpacked corpus; not saved, not in the 20 GB quota
# =====================================================================


In [ ]:
import os, shutil, subprocess
from pathlib import Path

for p in ("/tmp", "/kaggle/working"):
    t, u, f = shutil.disk_usage(p)
    print("%-16s %6.1f GB free of %6.1f GB" % (p, f / 2**30, t / 2**30))
if shutil.disk_usage("/tmp").free < 16 * 2**30:
    raise RuntimeError("/tmp has under 16 GB free; the unpacked corpus needs ~13 GB "
                       "plus the parquet cache")

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:  # noqa: BLE001
        print("[secrets] HF_TOKEN lookup failed:", type(e).__name__)
if not HF_TOKEN:
    raise RuntimeError("no HF_TOKEN - the corpus is gated")
os.environ["HF_TOKEN"] = HF_TOKEN

# The clone lives in /tmp too: anything under /kaggle/working counts against the
# 500-file output cap.
SRC = "/tmp/v2"
shutil.rmtree(SRC, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                "https://github.com/%s.git" % REPO, SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
!pip -q install -r requirements.txt 2>&1 | tail -2


In [ ]:
import time
t0 = time.time()
shards = "--max-shards %d" % MAX_SHARDS if MAX_SHARDS else ""
!python scripts/download_data.py --out {SCRATCH}/data {shards}
print("download: %.1f min" % ((time.time() - t0) / 60))


In [ ]:
t0 = time.time()
!python scripts/make_vad.py --data {SCRATCH}/data 2>&1 | tail -15
print("vad: %.1f min" % ((time.time() - t0) / 60))
t0 = time.time()
!python scripts/make_synthetic.py --data {SCRATCH}/data --out {SCRATCH}/synth -n {N_SYNTHETIC} 2>&1 | tail -3
print("synthetic: %.1f min" % ((time.time() - t0) / 60))


In [ ]:
t0 = time.time()
shutil.rmtree(OUT, ignore_errors=True)
!python scripts/pack_data.py --data {SCRATCH}/data --synth {SCRATCH}/synth --out {OUT} --move
rc = int(get_ipython().user_ns.get("_exit_code", 0) or 0)
if rc:
    raise RuntimeError("pack_data.py failed (exit %d)" % rc)
print("pack: %.1f min" % ((time.time() - t0) / 60))


In [ ]:
import json, collections
recs = [json.loads(l) for l in open(OUT + "/manifest.jsonl") if l.strip()]
syn = [json.loads(l) for l in open(OUT + "/synth/manifest.jsonl") if l.strip()]
print(len(recs), "clips |", collections.Counter(r["tier"] for r in recs))
print("with VAD:", sum("vad_off" in r for r in recs), "| synthetic:", len(syn))
files = [p for p in Path("/kaggle/working").rglob("*") if p.is_file()]
print("%d output files, %.2f GB" % (len(files), sum(p.stat().st_size for p in files) / 2**30))
assert len(files) < 450, "too many output files for Kaggle to save"
shutil.rmtree(SCRATCH, ignore_errors=True)
